In [ ]:
import numpy as np
import pandas as pd
import os
import pySMARTS
from datetime import datetime, timedelta

# Set the SMARTSPATH environment variable
SMARTSPATH = r'C:\Users\eravishankar\OneDrive - Cal Poly Pomona\Agrivoltaics paper\Final submission\Code\smarts-295-pc\SMARTS_295_PC'

# Location and atmospheric parameters
CMNT = 'Yuma_AZ'
ISPR = '1'
SPR = 1011.51
ALTIT = 0.
HEIGHT = 0.
LATIT = 32.69
LONGIT = -114.63
ZONE = -7
IATMOS = '1'
ATMOS = 'USSA'
IH2O = '1'
IO3 = '1'
IGAS = '1'
qCO2 = 370
ISPCTR = '1'
AEROS = 'S&F_RURAL'
ITURB = '0'
TAU5 = 0.1
IALBDX = '38'
ITILT = '1'
IALBDG = '38'
TILT = 37.
WAZIM = 180.
WLMN = 280
WLMX = 4000
SUNCOR = 1.0
SOLARC = 1367.0
IPRT = '2'
WPMN = 280
WPMX = 4000
INTVL = 1
IOTOT = 15
IOUT = '2 3 4 5 6 7 8 9 10 30 33 34 35 37 38'
ICIRC = '1'
SLOPE = 1.78
APERT = 2.91
LIMIT = 4.03
ISCAN = '0'
ILLUM = 0
IUV = 0
IMASS = '3'
RH = ''
TAIR = ''
SEASON = ''
TDAY = ''
W = ''
IALT = ''
AbO3 = ''
ILOAD = ''
ApCH2O = ''
ApCH4 = ''
ApCO = ''
ApHNO2 = ''
ApHNO3 = ''
ApNO = ''
ApNO2 = ''
ApNO3 = ''
ApO3 = ''
ApSO2 = ''
ALPHA1 = ''
ALPHA2 = ''
OMEGL = ''
GG = ''
BETA = ''
BCHUEP = ''
RANGE = ''
VISI = ''
TAU550 = ''
RHOX = ''
RHOG = ''
IFILT = ''
WV1 = ''
WV2 = ''
STEP = ''
FWHM = ''
ZENITH = ''
AZIM = ''
ELEV = ''
AMASS = ''
DSTEP = ''
# Output directory
output_dir = "Yuma_AZ_Spectral_Data"
os.makedirs(output_dir, exist_ok=True)

# Iterate over each hour of the year
start_time = datetime(2024, 1, 1, 0)
for hour_count in range(1, 8761):
    current_time = start_time + timedelta(hours=hour_count - 1)
    YEAR = current_time.year
    MONTH = current_time.month
    DAY = current_time.day
    HOUR = current_time.hour

    try:
        spectra_data = pySMARTS.SMARTSTimeLocation(CMNT, ISPR, SPR, ALTIT, HEIGHT, LATIT, IATMOS, ATMOS, RH, TAIR, SEASON, TDAY, IH2O, W, IO3, IALT, AbO3, IGAS, ILOAD, ApCH2O, ApCH4, ApCO, ApHNO2, ApHNO3, ApNO,ApNO2, ApNO3, ApO3, ApSO2, qCO2, ISPCTR, AEROS, ALPHA1, ALPHA2, OMEGL, GG, ITURB, TAU5, BETA, BCHUEP, RANGE, VISI, TAU550, IALBDX, RHOX, ITILT, IALBDG,TILT, WAZIM,  RHOG, WLMN, WLMX, SUNCOR, SOLARC, IPRT, WPMN, WPMX, INTVL, IOUT, ICIRC, SLOPE, APERT, LIMIT, ISCAN, IFILT, WV1, WV2, STEP, FWHM, ILLUM,IUV, IMASS, ZENITH, AZIM, ELEV, AMASS, YEAR, MONTH, DAY, HOUR, LONGIT, ZONE, DSTEP, SMARTSPATH)

        # Convert to DataFrame
        df = pd.DataFrame(spectra_data)

        # Define filename
        filename = os.path.join(output_dir, f"Yuma_AZ_{hour_count:04d}.xlsx")

        # Save to Excel
        df.to_excel(filename, index=False)
        print(f"Saved: {filename}")

    except Exception as e:
        print(f"Skipping hour {hour_count} ({current_time}) due to error: {e}")


In [1]:
import os
import pandas as pd
import numpy as np
from glob import glob

In [2]:
# Define directory
input_dir = "Yuma_AZ_Spectral_Data"

# Get list of all Excel files
excel_files = sorted(glob(os.path.join(input_dir, "*.xlsx")))

In [3]:
# Initialize dictionary to store spectral data
spectral_data = {}

# Read each file and store data
for file in excel_files:
    try:
        # Extract hour from filename
        hour_count = int(os.path.basename(file).split("_")[-1].split(".")[0])

        # Read Excel file
        df = pd.read_excel(file, engine="openpyxl")

        # First column is Wavelength
        wavelengths = df.iloc[:, 0].values

        # Iterate over all other columns (excluding wavelength)
        for col_idx in range(1, df.shape[1]):
            col_name = df.columns[col_idx]

            # Initialize matrix for this variable if not exists
            if col_name not in spectral_data:
                spectral_data[col_name] = {}

            # Store spectral values for the current hour
            spectral_data[col_name][hour_count] = df.iloc[:, col_idx].values

    except Exception as e:
        print(f"Error processing {file}: {e}")

# Convert dictionaries to matrices
output_dir = "Processed_Spectral_Matrices"
os.makedirs(output_dir, exist_ok=True)

In [4]:
for variable, data in spectral_data.items():
    try:
        # Convert to DataFrame (rows: wavelengths, columns: hours)
        matrix_df = pd.DataFrame.from_dict(data, orient="columns")

        # Insert Wavelength column
        matrix_df.insert(0, "Wavelength", wavelengths)

        # Save matrix as CSV
        output_file = os.path.join(output_dir, f"{variable}_matrix.csv")
        matrix_df.to_csv(output_file, index=False)
        print(f"Saved matrix for {variable}: {output_file}")

    except Exception as e:
        print(f"Error saving matrix for {variable}: {e}")


Saved matrix for Direct_normal_irradiance: Processed_Spectral_Matrices\Direct_normal_irradiance_matrix.csv
Saved matrix for Difuse_horizn_irradiance: Processed_Spectral_Matrices\Difuse_horizn_irradiance_matrix.csv
Saved matrix for Global_horizn_irradiance: Processed_Spectral_Matrices\Global_horizn_irradiance_matrix.csv
Saved matrix for Direct_horizn_irradiance: Processed_Spectral_Matrices\Direct_horizn_irradiance_matrix.csv
Saved matrix for Direct_tilted_irradiance: Processed_Spectral_Matrices\Direct_tilted_irradiance_matrix.csv
Saved matrix for Difuse_tilted_irradiance: Processed_Spectral_Matrices\Difuse_tilted_irradiance_matrix.csv
Saved matrix for Global_tilted_irradiance: Processed_Spectral_Matrices\Global_tilted_irradiance_matrix.csv
Saved matrix for Beam_normal_+circumsolar: Processed_Spectral_Matrices\Beam_normal_+circumsolar_matrix.csv
Saved matrix for Difuse_horiz-circumsolar: Processed_Spectral_Matrices\Difuse_horiz-circumsolar_matrix.csv
Saved matrix for Zonal_ground_reflect